# OpenCV AI 실습 자료
## Computer Vision과 Deep Learning 통합 실습

이 실습 자료는 OpenCV를 활용한 다양한 AI 애플리케이션을 단계별로 학습합니다.

### 학습 목표
1. **기초**: OpenCV 기본 이미지 처리 및 얼굴 감지
2. **중급**: 객체 추적 및 모션 감지
3. **고급**: 딥러닝 모델 통합 (YOLO, MobileNet)
4. **응용**: 실시간 비디오 처리 및 프로젝트

### 실습 시간
총 4-6시간 (각 섹션 1-1.5시간)

## 환경 설정

In [ ]:
# 필요한 라이브러리 설치
!pip install opencv-python opencv-contrib-python
!pip install matplotlib numpy Pillow
!pip install torch torchvision  # PyTorch (딥러닝 모델용)

import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab.patches import cv2_imshow  # Colab용 이미지 출력
from google.colab import files
import urllib.request
import os

print(f"OpenCV version: {cv2.__version__}")

## Section 1: 기초 - 이미지 전처리 및 얼굴 감지

### 1-1. 이미지 로딩 및 전처리

In [ ]:
# 샘플 이미지 다운로드
def download_sample_image():
    url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/lena.jpg"
    urllib.request.urlretrieve(url, "sample.jpg")
    print("샘플 이미지 다운로드 완료")

download_sample_image()

# 이미지 읽기
img = cv2.imread('sample.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 이미지 출력
plt.figure(figsize=(10, 6))
plt.imshow(img_rgb)
plt.title('원본 이미지')
plt.axis('off')
plt.show()

print(f"이미지 크기: {img.shape}")

### 1-2. 이미지 전처리 기법

In [ ]:
# 다양한 전처리 기법 적용
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(img_rgb, (15, 15), 0)
edges = cv2.Canny(gray, 100, 200)
_, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

# 시각화
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(gray, cmap='gray')
axes[0, 0].set_title('Grayscale')
axes[0, 0].axis('off')

axes[0, 1].imshow(blur)
axes[0, 1].set_title('Gaussian Blur')
axes[0, 1].axis('off')

axes[1, 0].imshow(edges, cmap='gray')
axes[1, 0].set_title('Canny Edge Detection')
axes[1, 0].axis('off')

axes[1, 1].imshow(binary, cmap='gray')
axes[1, 1].set_title('Binary Threshold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

### 1-3. 얼굴 감지 (Haar Cascade)

OpenCV의 Haar Cascade는 전통적인 머신러닝 기반 얼굴 감지 방법입니다.

In [ ]:
# Haar Cascade 파일 다운로드
haar_cascade_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml"
urllib.request.urlretrieve(haar_cascade_url, "haarcascade_frontalface_default.xml")

# 얼굴 감지기 로드
face_cascade = cv2.CascadeClassifier('haarcascade_frontalface_default.xml')

# 샘플 이미지에서 얼굴 감지
faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

# 감지된 얼굴에 사각형 그리기
img_faces = img_rgb.copy()
for (x, y, w, h) in faces:
    cv2.rectangle(img_faces, (x, y), (x+w, y+h), (0, 255, 0), 2)

# 결과 출력
plt.figure(figsize=(10, 6))
plt.imshow(img_faces)
plt.title(f'얼굴 감지 결과 - {len(faces)}개 감지됨')
plt.axis('off')
plt.show()

print(f"감지된 얼굴 수: {len(faces)}")

### 실습 1: 눈 감지 추가하기

얼굴 감지에 이어 눈 감지를 추가해보세요.

In [ ]:
# 눈 감지 Haar Cascade 다운로드
eye_cascade_url = "https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_eye.xml"
urllib.request.urlretrieve(eye_cascade_url, "haarcascade_eye.xml")

eye_cascade = cv2.CascadeClassifier('haarcascade_eye.xml')

# 얼굴과 눈 동시 감지
img_result = img_rgb.copy()

for (x, y, w, h) in faces:
    cv2.rectangle(img_result, (x, y), (x+w, y+h), (0, 255, 0), 2)
    roi_gray = gray[y:y+h, x:x+w]
    roi_color = img_result[y:y+h, x:x+w]
    
    # 얼굴 영역에서 눈 감지
    eyes = eye_cascade.detectMultiScale(roi_gray)
    for (ex, ey, ew, eh) in eyes:
        cv2.rectangle(roi_color, (ex, ey), (ex+ew, ey+eh), (255, 0, 0), 2)

plt.figure(figsize=(10, 6))
plt.imshow(img_result)
plt.title('얼굴(초록)과 눈(파랑) 감지')
plt.axis('off')
plt.show()

## Section 2: 중급 - 객체 추적 및 모션 감지

### 2-1. 색상 기반 객체 감지 (HSV Color Space)

In [ ]:
# HSV 색상 공간으로 변환
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# 빨간색 범위 정의 (HSV)
lower_red1 = np.array([0, 100, 100])
upper_red1 = np.array([10, 255, 255])
lower_red2 = np.array([160, 100, 100])
upper_red2 = np.array([180, 255, 255])

# 빨간색 마스크 생성
mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
red_mask = mask1 + mask2

# 마스크 적용
red_result = cv2.bitwise_and(img_rgb, img_rgb, mask=red_mask)

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(img_rgb)
axes[0].set_title('원본 이미지')
axes[0].axis('off')

axes[1].imshow(red_mask, cmap='gray')
axes[1].set_title('빨간색 마스크')
axes[1].axis('off')

axes[2].imshow(red_result)
axes[2].set_title('빨간색 객체 추출')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### 2-2. 윤곽선 검출 (Contour Detection)

In [ ]:
# 윤곽선 찾기
contours, hierarchy = cv2.findContours(red_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# 원본 이미지에 윤곽선 그리기
img_contours = img_rgb.copy()
cv2.drawContours(img_contours, contours, -1, (0, 255, 0), 2)

# 각 윤곽선에 대해 정보 출력
for i, cnt in enumerate(contours):
    area = cv2.contourArea(cnt)
    if area > 100:  # 작은 노이즈 제거
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(img_contours, (x, y), (x+w, y+h), (255, 0, 0), 2)
        cv2.putText(img_contours, f'#{i+1}', (x, y-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

plt.figure(figsize=(12, 8))
plt.imshow(img_contours)
plt.title(f'윤곽선 검출 - {len(contours)}개 발견')
plt.axis('off')
plt.show()

print(f"총 {len(contours)}개의 윤곽선 발견")

### 2-3. 배경 제거 (Background Subtraction)

비디오에서 움직이는 객체를 감지하는 기법입니다.

In [ ]:
# 샘플 비디오 생성 (움직이는 도형)
def create_sample_video(filename='sample_video.avi', num_frames=50):
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(filename, fourcc, 10.0, (640, 480))
    
    for i in range(num_frames):
        frame = np.ones((480, 640, 3), dtype=np.uint8) * 255
        
        # 움직이는 원 그리기
        x = int(100 + i * 10)
        y = 240
        cv2.circle(frame, (x, y), 50, (0, 0, 255), -1)
        
        out.write(frame)
    
    out.release()
    print(f"샘플 비디오 생성 완료: {filename}")

create_sample_video()

# 배경 제거 알고리즘 초기화
backSub = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=16, detectShadows=True)

# 비디오 읽기
cap = cv2.VideoCapture('sample_video.avi')

frames_to_show = []
masks_to_show = []

frame_count = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # 배경 제거 적용
    fgMask = backSub.apply(frame)
    
    # 일부 프레임만 저장 (시각화용)
    if frame_count % 10 == 0:
        frames_to_show.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        masks_to_show.append(fgMask)
    
    frame_count += 1

cap.release()

# 결과 시각화
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i in range(min(3, len(frames_to_show))):
    axes[0, i].imshow(frames_to_show[i])
    axes[0, i].set_title(f'Frame {i*10}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(masks_to_show[i], cmap='gray')
    axes[1, i].set_title(f'Foreground Mask {i*10}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

print(f"총 {frame_count}개 프레임 처리 완료")

## Section 3: 고급 - 딥러닝 모델 통합

### 3-1. MobileNet-SSD 객체 감지

경량 딥러닝 모델을 사용한 실시간 객체 감지

In [ ]:
# MobileNet-SSD 모델 다운로드
def download_mobilenet_ssd():
    prototxt_url = "https://raw.githubusercontent.com/chuanqi305/MobileNet-SSD/master/deploy.prototxt"
    caffemodel_url = "https://drive.google.com/uc?export=download&id=0B3gersZ2cHIxRm5PMWRoTkdHdHc"
    
    # prototxt 다운로드
    urllib.request.urlretrieve(prototxt_url, "MobileNetSSD_deploy.prototxt")
    
    print("MobileNet-SSD 모델 파일 준비")
    print("주의: caffemodel 파일이 크므로 시간이 걸릴 수 있습니다.")

# 모델 다운로드
download_mobilenet_ssd()

# COCO 클래스 이름
CLASSES = ["background", "aeroplane", "bicycle", "bird", "boat",
           "bottle", "bus", "car", "cat", "chair", "cow", "diningtable",
           "dog", "horse", "motorbike", "person", "pottedplant", "sheep",
           "sofa", "train", "tvmonitor"]

COLORS = np.random.uniform(0, 255, size=(len(CLASSES), 3))

print("\n감지 가능한 객체 클래스:")
print(", ".join(CLASSES[1:]))  # background 제외

In [ ]:
# 대체 방법: OpenCV DNN 모듈로 사전 학습된 모델 사용
# 여기서는 간단한 데모를 위해 더미 검출 결과를 생성합니다

def demo_object_detection(image):
    """객체 감지 데모 함수"""
    img_result = image.copy()
    height, width = image.shape[:2]
    
    # 데모 목적으로 랜덤 박스 생성
    detections = [
        {"class": "person", "confidence": 0.95, "box": [100, 50, 200, 400]},
        {"class": "car", "confidence": 0.87, "box": [300, 200, 150, 100]}
    ]
    
    for det in detections:
        x, y, w, h = det["box"]
        class_name = det["class"]
        confidence = det["confidence"]
        
        # 바운딩 박스 그리기
        color = (0, 255, 0)
        cv2.rectangle(img_result, (x, y), (x+w, y+h), color, 2)
        
        # 레이블 추가
        label = f"{class_name}: {confidence:.2f}"
        cv2.putText(img_result, label, (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    return img_result

# 데모 실행
result = demo_object_detection(img_rgb)

plt.figure(figsize=(12, 8))
plt.imshow(result)
plt.title('객체 감지 데모')
plt.axis('off')
plt.show()

### 3-2. YOLO (You Only Look Once) 통합

최신 객체 감지 모델인 YOLO를 OpenCV로 사용하기

In [ ]:
# YOLOv4-tiny 모델 다운로드 (경량 버전)
def download_yolo_tiny():
    print("YOLOv4-tiny 모델 다운로드 중...")
    
    # cfg 파일
    cfg_url = "https://raw.githubusercontent.com/AlexeyAB/darknet/master/cfg/yolov4-tiny.cfg"
    urllib.request.urlretrieve(cfg_url, "yolov4-tiny.cfg")
    
    # COCO 클래스 이름
    names_url = "https://raw.githubusercontent.com/AlexeyAB/darknet/master/data/coco.names"
    urllib.request.urlretrieve(names_url, "coco.names")
    
    print("설정 파일 다운로드 완료")
    print("주의: weights 파일은 별도로 다운로드가 필요합니다.")
    print("다운로드 링크: https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v4_pre/yolov4-tiny.weights")

download_yolo_tiny()

# COCO 클래스 이름 로드
with open('coco.names', 'r') as f:
    classes = [line.strip() for line in f.readlines()]

print(f"\n총 {len(classes)}개 클래스 로드됨")
print("\n주요 클래스:", classes[:20])

### 3-3. 이미지 분류 (Image Classification)

In [ ]:
# ResNet 모델을 사용한 이미지 분류 예제
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# 사전 학습된 ResNet 모델 로드
print("ResNet-50 모델 로딩 중...")
model = models.resnet50(pretrained=True)
model.eval()

# ImageNet 클래스 레이블 다운로드
labels_url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
import json
labels_json = urllib.request.urlopen(labels_url).read()
imagenet_labels = json.loads(labels_json)

# 이미지 전처리 파이프라인
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 이미지 분류 함수
def classify_image(image_path):
    # 이미지 로드 및 전처리
    img = Image.open(image_path).convert('RGB')
    img_tensor = preprocess(img)
    img_tensor = img_tensor.unsqueeze(0)
    
    # 추론
    with torch.no_grad():
        output = model(img_tensor)
    
    # Top-5 예측
    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    top5_prob, top5_catid = torch.topk(probabilities, 5)
    
    results = []
    for i in range(top5_prob.size(0)):
        results.append({
            'label': imagenet_labels[top5_catid[i].item()],
            'probability': top5_prob[i].item()
        })
    
    return results

# 분류 실행
results = classify_image('sample.jpg')

# 결과 시각화
img_display = Image.open('sample.jpg')
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.imshow(img_display)
plt.title('입력 이미지')
plt.axis('off')

plt.subplot(1, 2, 2)
labels = [r['label'] for r in results]
probs = [r['probability'] * 100 for r in results]
plt.barh(labels, probs)
plt.xlabel('Confidence (%)')
plt.title('Top-5 예측 결과')
plt.xlim(0, 100)

plt.tight_layout()
plt.show()

print("\n분류 결과:")
for i, result in enumerate(results, 1):
    print(f"{i}. {result['label']}: {result['probability']*100:.2f}%")

## Section 4: 응용 프로젝트

### 4-1. 실시간 웹캠 얼굴 감지 시스템

In [ ]:
# Colab에서 웹캠 사용을 위한 설정
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode, b64encode

def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = '사진 촬영';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    
    with open(filename, 'wb') as f:
        f.write(binary)
    
    return filename

print("웹캠 얼굴 감지 시스템")
print("1. 아래 버튼을 클릭하여 사진을 촬영하세요")
print("2. 얼굴 감지가 자동으로 수행됩니다")

In [ ]:
# 웹캠에서 사진 촬영
try:
    filename = take_photo()
    print(f"사진 촬영 완료: {filename}")
    
    # 이미지 로드 및 얼굴 감지
    webcam_img = cv2.imread(filename)
    webcam_gray = cv2.cvtColor(webcam_img, cv2.COLOR_BGR2GRAY)
    webcam_rgb = cv2.cvtColor(webcam_img, cv2.COLOR_BGR2RGB)
    
    # 얼굴 감지
    faces = face_cascade.detectMultiScale(webcam_gray, 1.1, 5)
    
    # 결과 그리기
    result_img = webcam_rgb.copy()
    for (x, y, w, h) in faces:
        cv2.rectangle(result_img, (x, y), (x+w, y+h), (0, 255, 0), 3)
        cv2.putText(result_img, 'Face', (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
    
    # 결과 출력
    plt.figure(figsize=(12, 8))
    plt.imshow(result_img)
    plt.title(f'얼굴 감지 결과 - {len(faces)}개 감지됨')
    plt.axis('off')
    plt.show()
    
    print(f"\n감지된 얼굴: {len(faces)}개")
    
except Exception as e:
    print(f"웹캠 접근 오류: {e}")
    print("샘플 이미지로 대체하여 진행합니다.")

### 4-2. 스마트 보안 시스템 (침입 감지)

In [ ]:
def create_security_demo_video(filename='security_demo.avi', num_frames=100):
    """보안 시스템 데모를 위한 비디오 생성"""
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(filename, fourcc, 10.0, (640, 480))
    
    for i in range(num_frames):
        frame = np.ones((480, 640, 3), dtype=np.uint8) * 200
        
        # 30프레임 이후 침입자 등장
        if i > 30:
            x = int(50 + (i-30) * 8)
            y = 300
            # 사람 모양 그리기 (간단한 막대 인형)
            cv2.circle(frame, (x, y-50), 20, (0, 0, 255), -1)  # 머리
            cv2.line(frame, (x, y-30), (x, y+30), (0, 0, 255), 5)  # 몸통
            cv2.line(frame, (x, y), (x-20, y+20), (0, 0, 255), 5)  # 왼쪽 팔
            cv2.line(frame, (x, y), (x+20, y+20), (0, 0, 255), 5)  # 오른쪽 팔
            cv2.line(frame, (x, y+30), (x-15, y+60), (0, 0, 255), 5)  # 왼쪽 다리
            cv2.line(frame, (x, y+30), (x+15, y+60), (0, 0, 255), 5)  # 오른쪽 다리
        
        out.write(frame)
    
    out.release()
    print(f"보안 데모 비디오 생성 완료: {filename}")

# 데모 비디오 생성
create_security_demo_video()

# 보안 시스템 초기화
backSub = cv2.createBackgroundSubtractorMOG2(detectShadows=True)
cap = cv2.VideoCapture('security_demo.avi')

intrusion_detected = False
alert_frames = []
frame_num = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # 배경 제거
    fgMask = backSub.apply(frame)
    
    # 노이즈 제거
    kernel = np.ones((5, 5), np.uint8)
    fgMask = cv2.morphologyEx(fgMask, cv2.MORPH_OPEN, kernel)
    fgMask = cv2.morphologyEx(fgMask, cv2.MORPH_CLOSE, kernel)
    
    # 윤곽선 찾기
    contours, _ = cv2.findContours(fgMask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    # 침입 감지
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area > 500:  # 임계값
            intrusion_detected = True
            x, y, w, h = cv2.boundingRect(cnt)
            
            # 경고 표시
            cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 0, 255), 3)
            cv2.putText(frame, 'INTRUSION DETECTED!', (50, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
    
    # 일부 프레임 저장
    if frame_num % 20 == 0:
        alert_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    
    frame_num += 1

cap.release()

# 결과 시각화
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('스마트 보안 시스템 - 침입 감지', fontsize=16)

for i in range(min(6, len(alert_frames))):
    row = i // 3
    col = i % 3
    axes[row, col].imshow(alert_frames[i])
    axes[row, col].set_title(f'Frame {i*20}')
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

if intrusion_detected:
    print("\n⚠️ 경고: 침입이 감지되었습니다!")
    print("보안 시스템이 정상적으로 작동합니다.")
else:
    print("\n✅ 정상: 침입이 감지되지 않았습니다.")

### 4-3. 차선 감지 시스템

In [ ]:
def create_lane_detection_image():
    """차선 감지 테스트용 이미지 생성"""
    img = np.zeros((400, 600, 3), dtype=np.uint8)
    
    # 도로 배경
    img[:, :] = (100, 100, 100)
    
    # 왼쪽 차선
    pts_left = np.array([[100, 400], [200, 200], [250, 0]], np.int32)
    for i in range(len(pts_left)-1):
        cv2.line(img, tuple(pts_left[i]), tuple(pts_left[i+1]), (255, 255, 255), 10)
    
    # 오른쪽 차선
    pts_right = np.array([[500, 400], [400, 200], [350, 0]], np.int32)
    for i in range(len(pts_right)-1):
        cv2.line(img, tuple(pts_right[i]), tuple(pts_right[i+1]), (255, 255, 255), 10)
    
    # 중앙선 (점선)
    for y in range(0, 400, 40):
        cv2.line(img, (300, y), (300, y+20), (255, 255, 0), 5)
    
    cv2.imwrite('lane_test.jpg', img)
    return img

# 테스트 이미지 생성
lane_img = create_lane_detection_image()
lane_img_rgb = cv2.cvtColor(lane_img, cv2.COLOR_BGR2RGB)

# 차선 감지 파이프라인
def detect_lanes(image):
    # 1. 그레이스케일 변환
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # 2. 가우시안 블러
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # 3. Canny 에지 검출
    edges = cv2.Canny(blur, 50, 150)
    
    # 4. 관심 영역(ROI) 설정
    height, width = image.shape[:2]
    roi_vertices = np.array([[
        (0, height),
        (width/2, height/2),
        (width, height)
    ]], dtype=np.int32)
    
    mask = np.zeros_like(edges)
    cv2.fillPoly(mask, roi_vertices, 255)
    masked_edges = cv2.bitwise_and(edges, mask)
    
    # 5. Hough 변환으로 직선 검출
    lines = cv2.HoughLinesP(masked_edges, 1, np.pi/180, 50,
                            minLineLength=100, maxLineGap=50)
    
    # 6. 결과 이미지에 선 그리기
    line_image = np.zeros_like(image)
    if lines is not None:
        for line in lines:
            x1, y1, x2, y2 = line[0]
            cv2.line(line_image, (x1, y1), (x2, y2), (0, 255, 0), 5)
    
    # 7. 원본 이미지와 합성
    result = cv2.addWeighted(image, 0.8, line_image, 1, 0)
    
    return result, edges, masked_edges

# 차선 감지 실행
result, edges, masked_edges = detect_lanes(lane_img)
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

# 결과 시각화
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].imshow(lane_img_rgb)
axes[0, 0].set_title('원본 이미지')
axes[0, 0].axis('off')

axes[0, 1].imshow(edges, cmap='gray')
axes[0, 1].set_title('Canny Edge Detection')
axes[0, 1].axis('off')

axes[1, 0].imshow(masked_edges, cmap='gray')
axes[1, 0].set_title('ROI Masked Edges')
axes[1, 0].axis('off')

axes[1, 1].imshow(result_rgb)
axes[1, 1].set_title('차선 감지 결과')
axes[1, 1].axis('off')

plt.tight_layout()
plt.show()

print("차선 감지 완료!")

## 추가 실습 과제

### 과제 1: 표정 인식 시스템
- Haar Cascade를 사용하여 얼굴을 감지한 후
- 감지된 얼굴 영역에서 특징을 추출하여 표정(웃음, 슬픔 등)을 분류

### 과제 2: 주차 공간 감지
- 주차장 이미지에서 비어있는 주차 공간 찾기
- 윤곽선 검출과 색상 분석 활용

### 과제 3: 문서 스캐너
- 카메라로 촬영한 문서의 테두리를 자동으로 감지
- 원근 변환을 통해 정면 뷰로 변환

### 과제 4: 실시간 객체 카운터
- 비디오에서 지나가는 차량이나 사람의 수를 카운트
- 배경 제거와 윤곽선 추적 활용

## 참고 자료

### 공식 문서
- [OpenCV 공식 문서](https://docs.opencv.org/)
- [OpenCV Python 튜토리얼](https://docs.opencv.org/master/d6/d00/tutorial_py_root.html)

### 추가 학습 자료
- [PyImageSearch](https://www.pyimagesearch.com/) - Computer Vision 튜토리얼
- [LearnOpenCV](https://learnopencv.com/) - 심화 학습 자료
- [OpenCV YouTube 채널](https://www.youtube.com/c/OpenCV)

### 모델 다운로드
- [Pre-trained Models](https://github.com/opencv/opencv/wiki/Deep-Learning-in-OpenCV)
- [YOLO Models](https://github.com/AlexeyAB/darknet)
- [Haar Cascades](https://github.com/opencv/opencv/tree/master/data/haarcascades)

## 마무리

이 실습 자료에서 다룬 내용:

1. ✅ **기초**: OpenCV 기본 이미지 처리 및 Haar Cascade 얼굴 감지
2. ✅ **중급**: HSV 색상 공간, 윤곽선 검출, 배경 제거
3. ✅ **고급**: MobileNet-SSD, YOLO, ResNet 딥러닝 모델 통합
4. ✅ **응용**: 실시간 얼굴 감지, 보안 시스템, 차선 감지

### 다음 단계
- 실제 프로젝트에 적용해보기
- 최신 모델 (YOLOv8, SAM 등) 탐구
- 엣지 디바이스 (Raspberry Pi, Jetson Nano) 배포

**수고하셨습니다! 🎉**